# In this script,
- Read raw IPEDS data and organize
- Read CLEANED CPS data (data_process.ipynb)
- calculate distance to the nearest college

In [1]:
import os
os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')

import pandas as pd
import numpy as np

In [2]:
ipeds_raw = pd.read_csv("dataset/raw_dataset/ipeds2024.csv")


In [3]:
ipeds_raw.head()

,unitid,institution name,year,HD2024.Postsecondary and Title IV institution indicator,HD2024.Longitude location of institution,HD2024.Latitude location of institution,DRVEF2024.Total enrollment
0,100654,Alabama A & M University,2024,Title IV postsecondary institution,-86.568502,34.783368,7295.0
1,100663,University of Alabama at Birmingham,2024,Title IV postsecondary institution,-86.799345,33.505697,20905.0
2,100690,Amridge University,2024,Title IV postsecondary institution,-86.174010,32.362609,639.0
3,100706,University of Alabama in Huntsville,2024,Title IV postsecondary institution,-86.640449,34.724557,8564.0
4,100724,Alabama State University,2024,Title IV postsecondary institution,-86.295677,32.364317,4081.0


In [4]:
# rename 
def rename_ipeds(df):
    df = df.copy()

    ipeds_map = {
        "institution name": "name",
        "HD2024.Postsecondary and Title IV institution indicator": "TitleIV",
        "HD2024.Longitude location of institution": "lon",
        "HD2024.Latitude location of institution": "lat",
        "DRVEF2024.Total  enrollment": "total_enrollment"
    }

    df.columns = df.columns.str.strip()
    df = df.rename(columns= ipeds_map)

    return df.copy()

ipeds_raw = rename_ipeds(ipeds_raw)

In [5]:
# read cps data (cleaned)
cps14 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps14.parquet")
cps13 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps13.parquet")
cps12 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps12.parquet")

In [6]:
cps12.head()

,person_num,log_wage,educ,exp,exp2,female,black,mv,age,birth_place,employed,PUMA,STATE,STATE_PUMA,same_home_as_14ys,same_home_as_13ys,same_home_as_12ys,mover,age_bins,pums_weight
3802,4,16.659324,0.0,20.0,400.0,1,0,7.0,26,1,1,00501,01,01_00501,1,1,1,1,30-39,15
3847,3,16.979841,12.0,8.0,64.0,0,1,5.0,26,1,1,01801,01,01_01801,1,1,1,1,30-39,189
3889,2,15.806328,12.0,0.0,0.0,0,1,5.0,18,1,1,01404,01,01_01404,1,1,1,1,20-29,155
3897,2,17.585665,18.0,17.0,289.0,1,0,7.0,41,1,1,00100,01,01_00100,1,1,1,1,50-59,26
3909,3,18.006397,16.0,16.0,256.0,0,0,7.0,38,1,1,02803,01,01_02803,1,1,1,1,40-49,96


In [7]:
# Compute nearest college per PUMA, then attach to each CPS person-level file
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd


def pick_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    raise KeyError(f"Column not found. candidates={candidates}")


def build_puma_college_distance(ipeds_df, puma_gdf):
    puma = puma_gdf.to_crs(4326).copy()

    state_col = pick_col(puma, ["STATEFP20", "STATEFP", "STATE"])
    puma_col = pick_col(puma, ["PUMACE20", "PUMA", "PUMACE"])

    puma["state_key"] = puma[state_col].astype(str).str.zfill(2)
    puma["puma_key"] = puma[puma_col].astype(str).str.zfill(5)

    puma_pts = puma[["state_key", "puma_key", "geometry"]].copy()
    # representative_point is guaranteed to lie inside polygon
    puma_pts["geometry"] = puma_pts.geometry.representative_point()

    lon_col = pick_col(ipeds_df, ["lon", "longitude", "HD2024.Longitude location of institution"])
    lat_col = pick_col(ipeds_df, ["lat", "latitude", "HD2024.Latitude location of institution"])
    unit_col = pick_col(ipeds_df, ["unitid", "UNITID"])

    colleges = ipeds_df[[unit_col, lon_col, lat_col]].copy()
    colleges.columns = ["nearest_college_unitid", "lon", "lat"]
    colleges["lon"] = pd.to_numeric(colleges["lon"], errors="coerce")
    colleges["lat"] = pd.to_numeric(colleges["lat"], errors="coerce")
    colleges = colleges.dropna(subset=["lon", "lat"]).copy()

    col_gdf = gpd.GeoDataFrame(
        colleges,
        geometry=gpd.points_from_xy(colleges["lon"], colleges["lat"]),
        crs="EPSG:4326"
    )

    nearest = gpd.sjoin_nearest(
        puma_pts.to_crs(5070),
        col_gdf.to_crs(5070),
        how="left",
        distance_col="dist_m"
    )

    out = nearest[["state_key", "puma_key", "nearest_college_unitid", "lon", "lat", "dist_m"]].copy()
    out = out.rename(columns={
        "lon": "nearest_college_lon",
        "lat": "nearest_college_lat"
    })
    out["nearest_college_dist_km"] = out["dist_m"] / 1000.0
    out = out.drop(columns=["dist_m"])

    return out


def attach_nearest_college_vars(cps_df, puma_dist_df):
    out = cps_df.copy()

    cps_puma_col = pick_col(out, ["PUMA", "puma"])
    out["puma_key"] = (
        pd.to_numeric(out[cps_puma_col], errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.zfill(5)
    )

    state_candidates = ["STATE", "state", "state_fips", "STATEFIP", "ST"]
    state_col = None
    for c in state_candidates:
        if c in out.columns:
            state_col = c
            break

    if state_col is None:
        raise KeyError("CPS data must have a state column for US-wide merge.")

    out["state_key"] = (
        pd.to_numeric(out[state_col], errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.zfill(2)
    )

    out = out.merge(puma_dist_df, on=["state_key", "puma_key"], how="left")
    out = out.drop(columns=["state_key", "puma_key"])

    return out

#Combine all sharpfiles
from pathlib import Path
shp_dir = Path("dataset/sharpfiles")

shp_files = list(shp_dir.glob("**/tl_2025_*_puma20.shp"))
print("Length:", len(shp_files))

gdfs = []
for f in shp_files:
    gdf = gpd.read_file(f, engine='pyogrio')
    gdfs.append(gdf)
puma_all_shp = pd.concat(gdfs, ignore_index=True)
#save as geodataframe
puma_all_shp = gpd.GeoDataFrame(puma_all_shp, geometry="geometry", crs=gdfs[0].crs)



puma_dist = build_puma_college_distance(ipeds_raw, puma_all_shp)

cps_map = {
    "12": cps12,
    "13": cps13,
    "14": cps14,
}

cps_with_dist = {}
for yr, df in cps_map.items():
    cps_with_dist[yr] = attach_nearest_college_vars(df, puma_dist)
    print(f"cps{yr} shape: {cps_with_dist[yr].shape}")
    print(cps_with_dist[yr][["nearest_college_unitid", "nearest_college_dist_km"]].head(2))

# Save as new files (keep original cleaned files unchanged)
out_dir = "dataset/cleaned_dataset"
for yr, df in cps_with_dist.items():
    out_path = f"{out_dir}/cleaned_cps{yr}_with_nearest_college.parquet"
    df.to_parquet(out_path)
    print("saved:", out_path)


Length: 49
cps12 shape: (130085, 24)
   nearest_college_unitid  nearest_college_dist_km
0                101277.0                17.088704
1                102313.0                 7.494981
cps13 shape: (139259, 24)
   nearest_college_unitid  nearest_college_dist_km
0                101277.0                17.088704
1                102313.0                 7.494981
cps14 shape: (148770, 24)
   nearest_college_unitid  nearest_college_dist_km
0                101277.0                17.088704
1                102313.0                 7.494981
saved: dataset/cleaned_dataset/cleaned_cps12_with_nearest_college.parquet
saved: dataset/cleaned_dataset/cleaned_cps13_with_nearest_college.parquet
saved: dataset/cleaned_dataset/cleaned_cps14_with_nearest_college.parquet


In [8]:
df = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps14_with_nearest_college.parquet")
df.head()

,person_num,log_wage,educ,exp,exp2,female,black,mv,age,birth_place,...,same_home_as_14ys,same_home_as_13ys,same_home_as_12ys,mover,age_bins,pums_weight,nearest_college_unitid,nearest_college_lon,nearest_college_lat,nearest_college_dist_km
0,4,16.659324,0.0,20.0,400.0,1,0,7.0,26,1,...,1,1,1,1,30-39,15,101277.0,-86.196931,34.278710,17.088704
1,3,16.979841,12.0,8.0,64.0,0,1,5.0,26,1,...,1,1,1,1,30-39,189,102313.0,-86.343057,32.350109,7.494981
2,2,15.806328,12.0,0.0,0.0,0,1,5.0,18,1,...,1,1,1,1,20-29,155,102049.0,-86.791799,33.464128,3.405524
3,1,16.964838,14.0,23.0,529.0,0,0,7.0,43,1,...,1,1,0,1,50-59,25,101736.0,-87.678089,34.739788,13.650303
4,2,17.585665,18.0,17.0,289.0,1,0,7.0,41,1,...,1,1,1,1,50-59,26,101736.0,-87.678089,34.739788,13.650303


In [9]:
print(cps14[["STATE", "PUMA"]].head())
print(cps14["STATE"].unique())

     STATE   PUMA
3802    01  00501
3847    01  01801
3889    01  01404
3896    01  00100
3897    01  00100
['01' '02' '04' '05' '06' '08' '09' '10' '11' '12' '13' '15' '16' '17'
 '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30' '31'
 '32' '33' '34' '35' '36' '37' '38' '39' '40' '41' '42' '44' '45' '46'
 '47' '48' '49' '50' '51' '53' '54' '55' '56']
